In [1]:
!pip install spacy feedparser
!python -m spacy download en_core_web_sm

     - 0 bytes ? 0:00:00
     - 0 bytes ? 0:00:00


ERROR: Wheel 'en-core-web-sm' located at C:\Users\georgin\AppData\Local\Temp\pip-unpack-isjjkcc6\en_core_web_sm-3.8.0-py3-none-any.whl is invalid.


In [5]:
import json, os
import spacy
from neo4j import GraphDatabase
from getpass import getpass

driver = GraphDatabase.driver("bolt://127.0.0.1:7687", auth=("neo4j", getpass("Neo4j password: ")))
DATABASE = "athmograph"
driver.verify_connectivity()

nlp = spacy.load("en_core_web_sm")
print("Connected to Neo4j and spaCy model loaded")

Neo4j password:  ········


Connected to Neo4j and spaCy model loaded


In [6]:
CITIES = ["Rotterdam", "Singapore", "Los Angeles", "Shanghai", "Mumbai", "Hamburg",
          "Busan", "Santos", "Antwerp", "Dubai", "Felixstowe"]

with driver.session(database=DATABASE) as session:
    port_codes = sorted(r["code"] for r in session.run("MATCH (p:Port) RETURN p.port_code AS code"))

port_city = dict(zip(port_codes, CITIES))
city_port = {city.lower(): code for code, city in port_city.items()}
print(port_city)

with driver.session(database=DATABASE) as session:
    for code, city in port_city.items():
        session.run("MATCH (p:Port {port_code: $c}) SET p.city = $city", c=code, city=city)

with open("port_city_map.json", "w") as f:
    json.dump(port_city, f, indent=2)

{'PORT01': 'Rotterdam', 'PORT02': 'Singapore', 'PORT03': 'Los Angeles', 'PORT04': 'Shanghai', 'PORT05': 'Mumbai', 'PORT06': 'Hamburg', 'PORT07': 'Busan', 'PORT08': 'Santos', 'PORT09': 'Antwerp', 'PORT10': 'Dubai', 'PORT11': 'Felixstowe'}


In [7]:
news = [
    "Dock workers at the Port of Rotterdam began a 48-hour strike on Monday, delaying container ships across Europe.",
    "A cyclone warning has forced the Port of Singapore to suspend loading operations.",
    "Congestion at the Port of Los Angeles is causing delays of up to two weeks for electronics shipments.",
    "Shanghai port reopened on Tuesday after a short closure caused by heavy fog.",
    "A new trade agreement between India and the UAE is expected to boost shipping volumes through Mumbai.",
    "Hamburg terminal operators warned of a shutdown after a cyber attack on port systems.",
    "Fuel prices rose slightly this week, with no impact on port operations in Busan.",
    "Flooding near the Port of Santos halted truck access and delayed coffee exports.",
    "Workers at Antwerp port called off the planned walkout after reaching an agreement.",
    "Heavy congestion and a crane breakdown are slowing vessels at the Port of Dubai.",
]

with open("news_sample.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(news))

with open("news_sample.txt", encoding="utf-8") as f:
    headlines = [line.strip() for line in f if line.strip()]

print(len(headlines), "headlines loaded")

10 headlines loaded


In [8]:
for text in headlines:
    doc = nlp(text)
    print(text)
    print("   entities:", [(e.text, e.label_) for e in doc.ents])
    print()

Dock workers at the Port of Rotterdam began a 48-hour strike on Monday, delaying container ships across Europe.
   entities: [('the Port of Rotterdam', 'LOC'), ('48-hour', 'TIME'), ('Monday', 'DATE'), ('Europe', 'LOC')]

A cyclone warning has forced the Port of Singapore to suspend loading operations.
   entities: [('the Port of Singapore', 'LOC')]

Congestion at the Port of Los Angeles is causing delays of up to two weeks for electronics shipments.
   entities: [('the Port of Los Angeles', 'FAC'), ('up to two weeks', 'DATE')]

Shanghai port reopened on Tuesday after a short closure caused by heavy fog.
   entities: [('Shanghai', 'GPE'), ('Tuesday', 'DATE')]

A new trade agreement between India and the UAE is expected to boost shipping volumes through Mumbai.
   entities: [('India', 'GPE'), ('UAE', 'ORG'), ('Mumbai', 'GPE')]

Hamburg terminal operators warned of a shutdown after a cyber attack on port systems.
   entities: [('Hamburg', 'GPE')]

Fuel prices rose slightly this week, with

In [9]:
LABELS = {"GPE", "LOC", "FAC", "ORG"}
HIGH = ["strike", "walkout", "shutdown", "closure", "suspend", "cyclone", "cyber attack", "flooding", "halted", "fire", "blockade"]
MEDIUM = ["congestion", "delay", "warning", "backlog", "breakdown", "shortage"]
RESOLVED = ["reopened", "called off", "no impact", "resumed"]

def score(text):
    t = text.lower()
    if not any(k in t for k in HIGH + MEDIUM):
        return 0.0
    if any(k in t for k in RESOLVED):
        return 0.1
    if any(k in t for k in HIGH):
        return 0.9
    return 0.6

events = {}
for text in headlines:
    risk = score(text)
    if risk == 0:
        continue
    doc = nlp(text)
    for ent in doc.ents:
        if ent.label_ in LABELS:
            for city, code in city_port.items():
                if city in ent.text.lower():
                    if risk > events.get(code, {}).get("risk", 0):
                        events[code] = {"risk": risk, "event": text}

for code, e in events.items():
    print(code, port_city[code], e["risk"], "-", e["event"])

PORT01 Rotterdam 0.9 - Dock workers at the Port of Rotterdam began a 48-hour strike on Monday, delaying container ships across Europe.
PORT02 Singapore 0.9 - A cyclone warning has forced the Port of Singapore to suspend loading operations.
PORT03 Los Angeles 0.6 - Congestion at the Port of Los Angeles is causing delays of up to two weeks for electronics shipments.
PORT04 Shanghai 0.1 - Shanghai port reopened on Tuesday after a short closure caused by heavy fog.
PORT06 Hamburg 0.9 - Hamburg terminal operators warned of a shutdown after a cyber attack on port systems.
PORT08 Santos 0.9 - Flooding near the Port of Santos halted truck access and delayed coffee exports.
PORT09 Antwerp 0.1 - Workers at Antwerp port called off the planned walkout after reaching an agreement.
PORT10 Dubai 0.6 - Heavy congestion and a crane breakdown are slowing vessels at the Port of Dubai.


In [10]:
with driver.session(database=DATABASE) as session:
    session.run("MATCH (n) WHERE n:Port OR n:Plant OR n:Customer SET n.risk_score = 0.0, n.last_event = null")

    for code, e in events.items():
        session.run(
            "MATCH (p:Port {port_code: $c}) SET p.risk_score = $r, p.last_event = $e, p.updated_at = datetime()",
            c=code, r=e["risk"], e=e["event"])

    session.run("""
        MATCH (pl:Plant)-[:HOSTED_AT]->(p:Port) WHERE p.risk_score > 0
        WITH pl, max(p.risk_score) AS m
        SET pl.risk_score = m * 0.7
    """)

    session.run("""
        MATCH (pl:Plant)-[:ORDERED_BY]->(c:Customer) WHERE pl.risk_score > 0
        WITH c, max(pl.risk_score) AS m
        SET c.risk_score = m * 0.5
    """)

print("Risk updated in Neo4j")

Risk updated in Neo4j


In [11]:
with driver.session(database=DATABASE) as session:
    print("PORTS AT RISK")
    for r in session.run("MATCH (p:Port) WHERE p.risk_score > 0 RETURN p.port_code AS code, p.city AS city, p.risk_score AS risk ORDER BY risk DESC"):
        print("  ", r["code"], r["city"], r["risk"])
    for label in ["Plant", "Customer"]:
        n = session.run("MATCH (n:" + label + ") WHERE n.risk_score > 0 RETURN count(n) AS n").single()["n"]
        print(label, "nodes affected:", n)

PORTS AT RISK
   PORT02 Singapore 0.9
   PORT01 Rotterdam 0.9
   PORT06 Hamburg 0.9
   PORT08 Santos 0.9
   PORT10 Dubai 0.6
   PORT03 Los Angeles 0.6
   PORT04 Shanghai 0.1
   PORT09 Antwerp 0.1
Plant nodes affected: 16
Customer nodes affected: 46


In [12]:
risk = {}
with driver.session(database=DATABASE) as session:
    result = session.run("""
        MATCH (n) WHERE n.risk_score IS NOT NULL AND n.risk_score > 0
        RETURN n.port_code AS port, n.plant_code AS plant, n.customer_id AS cust,
               n.risk_score AS risk, n.last_event AS event
    """)
    for r in result:
        key = r["port"] or r["plant"] or r["cust"]
        risk[key] = {"risk": round(r["risk"], 2), "event": r["event"]}

out = os.path.expanduser("~/atmograph-ui/public/risk.json")
with open(out, "w") as f:
    json.dump(risk, f)
print(len(risk), "risky nodes written to", out)

70 risky nodes written to C:\Users\georgin/atmograph-ui/public/risk.json


In [13]:
import feedparser
feed = feedparser.parse("https://news.google.com/rss/search?q=port+strike+OR+port+congestion&hl=en")
live = [e.title for e in feed.entries[:20]]
for h in live:
    print("-", h)

- Port operational updates from around the world (11 Nov - 17 Nov 2026) - Kuehne+Nagel
- Container peak season easing, port congestion ties up 10% of fleet - Seatrade Maritime News
- Port congestion has become a new normal, says MSC chief - Shipping Telegraph
- Asia-Europe pre-Lunar New Year peak hits amid poor reliability, port congestion - Journal of Commerce
- News Cargo surge and port congestion drive up Far East-Middle East rates - The Loadstar
- Maersk orders 26 container ships as Asian port congestion disrupts global supply chains - Automotive Logistics
- ICTSI put on notice over Durban port congestion crisis - Fruitnet
- Managing Port Congestion: Avoid Shipping Delays with Air Express - DHL
- Namport shares further clarity on allegations of Walvis Bay Port congestion Staff Reporter THE Namibia Ports Authority (Namport) has shed more light on allegations of congestion at the Port of Walvis Bay, explaining that the Walvis Bay Container Terminal is c - Instagram
- ‘Port congestion

In [14]:
import feedparser
feed = feedparser.parse("https://news.google.com/rss/search?q=port+strike+OR+port+congestion&hl=en")
live = [e.title for e in feed.entries[:20]]
for h in live:
    print("-", h)

- Port operational updates from around the world (11 Nov - 17 Nov 2026) - Kuehne+Nagel
- Container peak season easing, port congestion ties up 10% of fleet - Seatrade Maritime News
- Port congestion has become a new normal, says MSC chief - Shipping Telegraph
- Asia-Europe pre-Lunar New Year peak hits amid poor reliability, port congestion - Journal of Commerce
- News Cargo surge and port congestion drive up Far East-Middle East rates - The Loadstar
- Maersk orders 26 container ships as Asian port congestion disrupts global supply chains - Automotive Logistics
- ICTSI put on notice over Durban port congestion crisis - Fruitnet
- Managing Port Congestion: Avoid Shipping Delays with Air Express - DHL
- Namport shares further clarity on allegations of Walvis Bay Port congestion Staff Reporter THE Namibia Ports Authority (Namport) has shed more light on allegations of congestion at the Port of Walvis Bay, explaining that the Walvis Bay Container Terminal is c - Instagram
- ‘Port congestion